# NASA Phase 2 Data Quality and Cycle-Semantics Audit

## tl;dr

The full scan covered **26 batteries and 53,529,296 rows**. It found **9,927** maximal contiguous discharge runs (`mode == -1`), of which **9,612** pass the provisional completeness gate. Lifetime-relative `time` has **zero resets**, and discharge voltage/current have **zero missing values**. Three batteries contain the literal string `"  NAN"` in `temperature_battery`; the worst per-file invalid rate is **0.0229%**, so canonicalization must coerce it to missing and preserve an availability mask. The data pass the project's primary-source gate, but reference discharges alone cannot supply three calibration cycles for every battery (battery20 has only two); calibrated Q-ref must therefore follow the design rule of the earliest three valid complete discharges rather than requiring reference missions exclusively.

## Context & Methods

The intended grain is one timestamped battery-pack observation. Downstream use is leakage-safe causal modeling. The source is `data/raw/battery_alt_dataset.zip`, verified in Phase 1.

### Key Assumptions

- Only members under `battery_alt_dataset/` whose names match `batteryNN.csv` are physical data files; `__MACOSX` members are metadata.
- Field units and categorical meanings come from the archive's root README.
- A candidate discharge cycle is one maximal contiguous run with `mode == -1`; `start_time` is a cycle-day marker, not a cycle identifier.
- Capacity is diagnostic-only here and is integrated from `current_load` over `time` for each discharge run.

In [1]:
from collections import Counter
from pathlib import Path, PurePosixPath
import json
import re
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display

ARCHIVE = Path("data/raw/battery_alt_dataset.zip")
OUTPUT_JSON = Path("data/manifests/nasa_phase2_data_quality.json")
OUTPUT_CSV = Path("data/manifests/nasa_phase2_battery_profile.csv")
CHUNK_ROWS = 250_000
BATTERY_PATTERN = re.compile(r"battery\d{2}\.csv$")

## Data

### 1. Confirm documented meanings and physical members

In [2]:
with zipfile.ZipFile(ARCHIVE) as archive:
    root_readme = archive.read("battery_alt_dataset/README.txt").decode("utf-8")
    battery_members = sorted(
        name for name in archive.namelist()
        if name.startswith("battery_alt_dataset/")
        and BATTERY_PATTERN.search(PurePosixPath(name).name)
    )

print(root_readme[:3_500])
print(f"Physical battery CSV members: {len(battery_members)}")
print(battery_members)

An accelerated Life Testing Dataset for Lithium-Ion Batteries with Constant and Variable Loading Conditions:

The dataset repository is organized into three main folders, each containing one group of life cycled battery packs. 
Within each folder individual battery packs own their dedicated csv file for continuous data logging, which are named with their respective battery pack number.

The folders are named: 

- regular_alt_batteries: Containing one csv file for each battery pack cycled at the same load level or load range throughout lifetime
- recommissioned_batteries: Containing one csv file for each battery pack cycled at different load levels at varying life stages
- second_life_batteries: Containing one csv file for each second life battery pack cycled at constant current througout the second life 

The columns in each csv file contain the following data with the provided units: 

The following columns contain data throughout the cycling process: 

- start time: [mm:dd:yyyy hh:mm

### 2. Stream every physical CSV

The scan is chunked, deterministic, and bounded in memory. It profiles completeness, monotonic time, categorical values, and every contiguous discharge run. No full archive extraction occurs.

In [3]:
expected_columns = [
    "start_time", "time", "mode", "voltage_charger", "temperature_battery",
    "voltage_load", "current_load", "temperature_mosfet",
    "temperature_resistor", "mission_type",
]

battery_profiles = []
discharge_runs = []
mode_counts = Counter()
mission_counts = Counter()
invalid_value_examples = {}

with zipfile.ZipFile(ARCHIVE) as archive:
    for member in battery_members:
        battery_id = PurePosixPath(member).stem
        row_count = 0
        null_counts = Counter()
        numeric_invalid_counts = Counter()
        discharge_rows = 0
        discharge_null_current = 0
        discharge_null_voltage = 0
        time_resets = 0
        non_increasing_time = 0
        start_time_values = set()
        sampled_dt = []
        run_state = {}
        current_run_id = -1
        previous = None

        with archive.open(member) as source:
            for chunk in pd.read_csv(source, chunksize=CHUNK_ROWS):
                assert list(chunk.columns) == expected_columns
                row_count += len(chunk)
                null_counts.update(chunk.isna().sum().astype(int).to_dict())
                mode_labels = chunk["mode"].map(lambda value: "missing" if pd.isna(value) else str(float(value)))
                mission_labels = chunk["mission_type"].map(lambda value: "missing" if pd.isna(value) else str(float(value)))
                mode_counts.update(mode_labels.value_counts().to_dict())
                mission_counts.update(mission_labels.value_counts().to_dict())
                for column in expected_columns[1:]:
                    numeric = pd.to_numeric(chunk[column], errors="coerce")
                    invalid = chunk[column].notna() & numeric.isna()
                    numeric_invalid_counts[column] += int(invalid.sum())
                    if invalid.any():
                        key = f"{battery_id}:{column}"
                        examples = invalid_value_examples.setdefault(key, [])
                        for value in chunk.loc[invalid, column].astype(str).unique():
                            if value not in examples and len(examples) < 10:
                                examples.append(value)
                start_time_values.update(chunk["start_time"].dropna().astype(str).unique())

                mode = chunk["mode"].to_numpy(dtype=float)
                mission = chunk["mission_type"].fillna(-999.0).to_numpy(dtype=float)
                time_values = chunk["time"].to_numpy(dtype=float)
                current = chunk["current_load"].to_numpy(dtype=float)
                voltage = chunk["voltage_load"].to_numpy(dtype=float)

                previous_time = np.empty(len(chunk), dtype=float)
                previous_mode = np.empty(len(chunk), dtype=float)
                previous_mission = np.empty(len(chunk), dtype=float)
                previous_current = np.empty(len(chunk), dtype=float)
                if previous is None:
                    previous_time[0] = np.nan
                    previous_mode[0] = np.nan
                    previous_mission[0] = np.nan
                    previous_current[0] = np.nan
                else:
                    previous_time[0], previous_mode[0], previous_mission[0], previous_current[0] = previous
                previous_time[1:] = time_values[:-1]
                previous_mode[1:] = mode[:-1]
                previous_mission[1:] = mission[:-1]
                previous_current[1:] = current[:-1]

                dt = time_values - previous_time
                time_resets += int(np.sum(dt < 0))
                non_increasing_time += int(np.sum(dt <= 0))
                if len(sampled_dt) < 10_000:
                    valid_dt = dt[np.isfinite(dt) & (dt > 0)]
                    sampled_dt.extend(valid_dt[: 10_000 - len(sampled_dt)].tolist())

                changed = (mode != previous_mode) | (mission != previous_mission)
                if previous is None:
                    changed[0] = True
                run_ids = np.cumsum(changed, dtype=np.int64) + current_run_id
                current_run_id = int(run_ids[-1])

                discharge_mask = mode == -1
                discharge_rows += int(discharge_mask.sum())
                discharge_null_current += int(np.isnan(current[discharge_mask]).sum())
                discharge_null_voltage += int(np.isnan(voltage[discharge_mask]).sum())

                same_run = ~changed
                integration = np.zeros(len(chunk), dtype=float)
                integrable = (
                    discharge_mask & same_run & np.isfinite(dt) & (dt >= 0)
                    & np.isfinite(current) & np.isfinite(previous_current)
                )
                integration[integrable] = (
                    0.5 * (current[integrable] + previous_current[integrable])
                    * dt[integrable] / 3600.0
                )

                discharge_frame = pd.DataFrame({
                    "run_id": run_ids[discharge_mask],
                    "time": time_values[discharge_mask],
                    "mission": mission[discharge_mask],
                    "current": current[discharge_mask],
                    "voltage": voltage[discharge_mask],
                    "increment_ah": integration[discharge_mask],
                })
                for run_id, group in discharge_frame.groupby("run_id", sort=False):
                    valid_voltage = group.loc[group["voltage"] > 1.0, "voltage"]
                    update = {
                        "battery_id": battery_id,
                        "run_id": int(run_id),
                        "mission_type": None if group["mission"].iloc[0] == -999 else int(group["mission"].iloc[0]),
                        "samples": int(len(group)),
                        "start_s": float(group["time"].iloc[0]),
                        "end_s": float(group["time"].iloc[-1]),
                        "delivered_ah": float(group["increment_ah"].sum()),
                        "current_min_a": float(group["current"].min()),
                        "current_max_a": float(group["current"].max()),
                        "voltage_start_v": float(valid_voltage.iloc[0]) if len(valid_voltage) else None,
                        "voltage_end_v": float(valid_voltage.iloc[-1]) if len(valid_voltage) else None,
                    }
                    if run_id not in run_state:
                        run_state[run_id] = update
                    else:
                        state = run_state[run_id]
                        state["samples"] += update["samples"]
                        state["end_s"] = update["end_s"]
                        state["delivered_ah"] += update["delivered_ah"]
                        state["current_min_a"] = min(state["current_min_a"], update["current_min_a"])
                        state["current_max_a"] = max(state["current_max_a"], update["current_max_a"])
                        if state["voltage_start_v"] is None:
                            state["voltage_start_v"] = update["voltage_start_v"]
                        if update["voltage_end_v"] is not None:
                            state["voltage_end_v"] = update["voltage_end_v"]

                previous = (time_values[-1], mode[-1], mission[-1], current[-1])

        file_runs = list(run_state.values())
        for run in file_runs:
            run["duration_s"] = run["end_s"] - run["start_s"]
            run["voltage_drop_v"] = (
                run["voltage_start_v"] - run["voltage_end_v"]
                if run["voltage_start_v"] is not None and run["voltage_end_v"] is not None
                else None
            )
        discharge_runs.extend(file_runs)
        mission_run_counts = Counter(run["mission_type"] for run in file_runs)
        battery_profiles.append({
            "battery_id": battery_id,
            "member": member,
            "rows": row_count,
            "start_time_days": len(start_time_values),
            "time_resets": time_resets,
            "non_increasing_time": non_increasing_time,
            "median_sample_interval_s": float(np.median(sampled_dt)),
            "discharge_rows": discharge_rows,
            "reference_discharge_runs": mission_run_counts.get(0, 0),
            "regular_discharge_runs": mission_run_counts.get(1, 0),
            "other_discharge_runs": sum(v for k, v in mission_run_counts.items() if k not in {0, 1}),
            "discharge_current_null_rate": discharge_null_current / max(discharge_rows, 1),
            "discharge_voltage_null_rate": discharge_null_voltage / max(discharge_rows, 1),
            "temperature_battery_null_rate": null_counts["temperature_battery"] / row_count,
            "temperature_battery_invalid_rate": numeric_invalid_counts["temperature_battery"] / row_count,
        })

battery_profile = pd.DataFrame(battery_profiles).sort_values("battery_id").reset_index(drop=True)
discharge_profile = pd.DataFrame(discharge_runs)
print(f"Rows scanned: {battery_profile['rows'].sum():,}")
print(f"Discharge runs: {len(discharge_profile):,}")

/var/folders/qd/p4zzw2sd3cqg75mj4_4vksbh0000gn/T/ipykernel_19202/3634996093.py:31: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(source, chunksize=CHUNK_ROWS):


/var/folders/qd/p4zzw2sd3cqg75mj4_4vksbh0000gn/T/ipykernel_19202/3634996093.py:31: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(source, chunksize=CHUNK_ROWS):


/var/folders/qd/p4zzw2sd3cqg75mj4_4vksbh0000gn/T/ipykernel_19202/3634996093.py:31: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(source, chunksize=CHUNK_ROWS):


Rows scanned: 53,529,296
Discharge runs: 9,927


## Results

### 3. File-level completeness and timing

In [4]:
display(battery_profile)
print("Mode counts:", dict(mode_counts))
print("Mission counts (NaN is outside load-board discharge):", dict(mission_counts))

,battery_id,member,rows,start_time_days,time_resets,non_increasing_time,median_sample_interval_s,discharge_rows,reference_discharge_runs,regular_discharge_runs,other_discharge_runs,discharge_current_null_rate,discharge_voltage_null_rate,temperature_battery_null_rate,temperature_battery_invalid_rate
0,battery00,battery_alt_dataset/regular_alt_batteries/batt...,1101244,17,0,0,0.948,106006,9,166,0,0.0,0.0,0.0,0.000000e+00
1,battery01,battery_alt_dataset/regular_alt_batteries/batt...,4918193,74,0,0,0.959,748968,44,847,0,0.0,0.0,0.0,0.000000e+00
2,battery02,battery_alt_dataset/recommissioned_batteries/b...,2069667,27,0,0,0.947,260562,20,396,0,0.0,0.0,0.0,0.000000e+00
3,battery03,battery_alt_dataset/recommissioned_batteries/b...,2161444,36,0,0,0.970,343327,22,419,0,0.0,0.0,0.0,4.626537e-07
4,battery10,battery_alt_dataset/regular_alt_batteries/batt...,907341,14,0,0,0.948,89844,8,140,0,0.0,0.0,0.0,0.000000e+00
5,battery11,battery_alt_dataset/regular_alt_batteries/batt...,4914864,74,0,0,0.951,759722,43,841,0,0.0,0.0,0.0,0.000000e+00
6,battery12,battery_alt_dataset/recommissioned_batteries/b...,3041084,42,0,0,0.965,490685,33,600,0,0.0,0.0,0.0,0.000000e+00
7,battery13,battery_alt_dataset/second_life_batteries/batt...,1736149,39,0,0,0.970,240239,17,307,0,0.0,0.0,0.0,0.000000e+00
8,battery20,battery_alt_dataset/regular_alt_batteries/batt...,207391,5,0,0,0.950,18927,2,31,0,0.0,0.0,0.0,0.000000e+00
9,battery21,battery_alt_dataset/regular_alt_batteries/batt...,530718,7,0,0,0.950,46680,4,78,0,0.0,0.0,0.0,0.000000e+00


Mode counts: {'1.0': 26345098, '0.0': 19822183, '-1.0': 7362015}
Mission counts (NaN is outside load-board discharge): {'missing': 46167281, '1.0': 5954568, '0.0': 1407447}


### 4. Discharge-run validity and recoverable capacity

In [5]:
discharge_profile["valid_candidate"] = (
    (discharge_profile["duration_s"] >= 60)
    & (discharge_profile["samples"] >= 50)
    & (discharge_profile["delivered_ah"] > 0)
    & (discharge_profile["voltage_drop_v"] > 0)
)
mission_summary = (
    discharge_profile.groupby("mission_type", dropna=False)
    .agg(
        runs=("run_id", "size"),
        valid_runs=("valid_candidate", "sum"),
        duration_median_s=("duration_s", "median"),
        capacity_median_ah=("delivered_ah", "median"),
        capacity_min_ah=("delivered_ah", "min"),
        capacity_max_ah=("delivered_ah", "max"),
        voltage_drop_median_v=("voltage_drop_v", "median"),
    )
)
valid_by_battery = discharge_profile.groupby("battery_id")["valid_candidate"].sum().rename("valid_discharge_runs")
reference_by_battery = (
    discharge_profile.loc[discharge_profile["mission_type"] == 0]
    .groupby("battery_id")["valid_candidate"].sum().rename("valid_reference_runs")
)
quality_gate = (
    battery_profile[["battery_id", "time_resets", "discharge_current_null_rate", "discharge_voltage_null_rate"]]
    .merge(valid_by_battery, on="battery_id", how="left")
    .merge(reference_by_battery, on="battery_id", how="left")
    .fillna({"valid_discharge_runs": 0, "valid_reference_runs": 0})
)
display(mission_summary)
display(quality_gate)
print("All batteries have >=3 valid discharges:", bool((quality_gate["valid_discharge_runs"] >= 3).all()))
print("All batteries have >=3 valid reference discharges:", bool((quality_gate["valid_reference_runs"] >= 3).all()))

,runs,valid_runs,duration_median_s,capacity_median_ah,capacity_min_ah,capacity_max_ah,voltage_drop_median_v
mission_type,,,,,,,
0,496,481,2925.5765,2.047698,0.000257,2.492233,3.381
1,9431,9131,594.8060,2.039725,0.000000,2.486076,3.589


,battery_id,time_resets,discharge_current_null_rate,discharge_voltage_null_rate,valid_discharge_runs,valid_reference_runs
0,battery00,0,0.0,0.0,170,9
1,battery01,0,0.0,0.0,862,42
2,battery02,0,0.0,0.0,415,20
3,battery03,0,0.0,0.0,418,21
4,battery10,0,0.0,0.0,145,7
5,battery11,0,0.0,0.0,865,43
6,battery12,0,0.0,0.0,622,32
7,battery13,0,0.0,0.0,323,17
8,battery20,0,0.0,0.0,29,2
9,battery21,0,0.0,0.0,81,4


All batteries have >=3 valid discharges: True
All batteries have >=3 valid reference discharges: False


### 5. Persist compact audit evidence

In [6]:
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
battery_profile.to_csv(OUTPUT_CSV, index=False)
summary = {
    "archive": str(ARCHIVE),
    "battery_count": int(len(battery_profile)),
    "row_count": int(battery_profile["rows"].sum()),
    "discharge_run_count": int(len(discharge_profile)),
    "valid_discharge_run_count": int(discharge_profile["valid_candidate"].sum()),
    "all_batteries_have_three_valid_discharges": bool((quality_gate["valid_discharge_runs"] >= 3).all()),
    "all_batteries_have_three_valid_reference_discharges": bool((quality_gate["valid_reference_runs"] >= 3).all()),
    "time_reset_count": int(battery_profile["time_resets"].sum()),
    "max_discharge_current_null_rate": float(battery_profile["discharge_current_null_rate"].max()),
    "max_discharge_voltage_null_rate": float(battery_profile["discharge_voltage_null_rate"].max()),
    "max_temperature_battery_invalid_rate": float(battery_profile["temperature_battery_invalid_rate"].max()),
    "invalid_value_examples": invalid_value_examples,
    "median_sample_interval_s_range": [
        float(battery_profile["median_sample_interval_s"].min()),
        float(battery_profile["median_sample_interval_s"].max()),
    ],
    "mode_counts": {str(k): int(v) for k, v in mode_counts.items()},
    "mission_counts": {str(k): int(v) for k, v in mission_counts.items()},
    "mission_summary": {
        str(index): {
            key: (int(value) if key in {"runs", "valid_runs"} else float(value))
            for key, value in row.items()
        }
        for index, row in mission_summary.to_dict(orient="index").items()
    },
}
OUTPUT_JSON.write_text(json.dumps(summary, indent=2) + "\n")
print(json.dumps(summary, indent=2))

{
  "archive": "data/raw/battery_alt_dataset.zip",
  "battery_count": 26,
  "row_count": 53529296,
  "discharge_run_count": 9927,
  "valid_discharge_run_count": 9612,
  "all_batteries_have_three_valid_discharges": true,
  "all_batteries_have_three_valid_reference_discharges": false,
  "time_reset_count": 0,
  "max_discharge_current_null_rate": 0.0,
  "max_discharge_voltage_null_rate": 0.0,
  "max_temperature_battery_invalid_rate": 0.0002293593340878195,
  "invalid_value_examples": {
    "battery03:temperature_battery": [
      "  NAN"
    ],
    "battery22:temperature_battery": [
      "  NAN"
    ],
    "battery30:temperature_battery": [
      "  NAN"
    ]
  },
  "median_sample_interval_s_range": [
    0.9459999999999127,
    0.9760000000005675
  ],
  "mode_counts": {
    "1.0": 26345098,
    "0.0": 19822183,
    "-1.0": 7362015
  },
  "mission_counts": {
    "missing": 46167281,
    "1.0": 5954568,
    "0.0": 1407447
  },
  "mission_summary": {
    "0": {
      "runs": 496,
      "v

## Takeaways

- The README is authoritative for units and categorical meanings; no semantic mapping is inferred from column names alone.
- `time` is globally continuous in seconds across each battery lifetime. Any future observed reset is a build-stopping integrity failure.
- A canonical cycle is a maximal contiguous `mode == -1` run, with `mission_type` retained as load-protocol metadata. `start_time` is a cycle-day marker, not a unique cycle key.
- Capacity is recoverable by causal trapezoidal integration of positive `current_load` over `time`; the observed median is about 2.04 Ah for both reference and regular missions.
- Reference discharges (`mission_type == 0`) are useful audit anchors, but the calibrated protocol must use the earliest three valid complete discharges because not every battery has three valid reference runs.
- The literal `"  NAN"` temperature values must be coerced to missing; downstream filling must use training-only statistics while preserving `temperature_available`.
- Phase 2 automation must enforce the validity gate and reject batteries with fewer than three complete discharges.